# Slither vs Claude Haiku

Frozen corpus of 723 Solidity files, built in `01_data_engineering.ipynb`.
Runs both tools, scores them, writes `data/stage3_scored/`.

→ Corpus, compiler map and preprocessed sources are already on disk. Nothing here rebuilds them.

## Setup

In [ ]:
import os, sys, json, re, time, hashlib, subprocess
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd

os.chdir("/Users/mk/Desktop/mk/AuditMind")
for p in ["/Users/mk/.foundry/bin"]:
    if p not in os.environ["PATH"]:
        os.environ["PATH"] = f"{p}:{os.environ['PATH']}"

STAGE2 = Path("data/stage2_results"); STAGE2.mkdir(parents=True, exist_ok=True)
STAGE3 = Path("data/stage3_scored");  STAGE3.mkdir(parents=True, exist_ok=True)

INCLUDE_OZ    = os.path.abspath("node_modules_oz/node_modules")
INCLUDE_OZ_V4 = os.path.abspath("node_modules_oz_v4/node_modules")

CATEGORIES = ["reentrancy", "access_control", "unchecked_low_level_calls",
              "denial_of_service", "time_manipulation"]

manifest      = json.load(open("data/manifests/stage1_manifest.json"))
compiler_map  = json.load(open("data/manifests/compiler_map.json"))
corpus_record = json.load(open("data/manifests/frozen_corpus_record.json"))

cj = lambda o: json.dumps(o, sort_keys=True, separators=(",", ":"))
h = hashlib.sha256(cj(sorted(manifest, key=lambda r: r["path"])).encode()).hexdigest()
assert h == corpus_record["corpus_hash"], "corpus mismatch"
print(f"{len(manifest)} files, hash verified")

## Slither

→ Scans the original nested sources, not the flat Haiku copies — sibling imports need the real tree.
→ `--base-path` only exists from solc 0.6, `--include-path` only from 0.8. Passing them to 0.4/0.5 errors.
→ `--json -` swallows error text on failure, so failures are re-run in plain mode to capture it.

In [ ]:
def scan_target(e):
    if e["source"] in ("dappscan", "clean"):
        rel = os.path.basename(e["path"]).replace("__", "/")
        if os.path.exists(rel):
            root = rel.split("/")[0]
            return os.path.relpath(rel, root), os.path.abspath(root)
    p = os.path.abspath(e["path"])
    return os.path.basename(p), os.path.dirname(p)

_cur = {"v": None}
def use_solc(v):
    if _cur["v"] != v:
        subprocess.run(["solc-select", "use", v], capture_output=True, text=True)
        _cur["v"] = v

def solc_flags(series):
    f = ["--allow-paths", "."]
    if series in ("0.6", "0.7", "0.8"): f += ["--base-path", "."]
    if series == "0.8":                 f += ["--include-path", INCLUDE_OZ]
    return f

def run_slither(e, timeout=120):
    fname, base = scan_target(e)
    ver = compiler_map.get(e["path"])
    out = {"path": e["path"], "source": e["source"],
           "label_type": e["label_type"], "compiler": ver}
    if ver is None:
        return {**out, "status": "no_compiler", "wallclock_s": 0,
                "returncode": None, "raw_stdout": "", "raw_stderr": ""}
    use_solc(ver)
    cmd = ["slither", fname, "--solc-args", " ".join(solc_flags(".".join(ver.split(".")[:2])))]
    t0 = time.time()
    try:
        r = subprocess.run(cmd + ["--json", "-"], capture_output=True, text=True,
                           timeout=timeout, cwd=base)
        el = time.time() - t0
        if r.stdout.strip().startswith("{"):
            return {**out, "status": "ok", "wallclock_s": el, "returncode": r.returncode,
                    "raw_stdout": r.stdout, "raw_stderr": r.stderr[-5000:]}
        r2 = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, cwd=base)
        return {**out, "status": "error", "wallclock_s": el, "returncode": r2.returncode,
                "raw_stdout": r2.stdout, "raw_stderr": r2.stderr[-5000:]}
    except subprocess.TimeoutExpired:
        return {**out, "status": "timeout", "wallclock_s": timeout,
                "returncode": None, "raw_stdout": "", "raw_stderr": "TIMEOUT"}

In [ ]:
t0 = time.time()
slither_results = [run_slither(r) for r in manifest]
print(f"{time.time()-t0:.0f}s", Counter(r["status"] for r in slither_results))
assert len(slither_results) == len(manifest)
json.dump(slither_results, open(STAGE2 / "slither_results.json", "w"))
print(pd.DataFrame(slither_results).groupby(["source","status"]).size().unstack(fill_value=0))

## Slither failures

→ All 723 compile. These are Slither's own analysis failing after a successful compile.
→ Excluded from scoring for both tools, so Haiku gets no credit on files its competitor never saw.

In [ ]:
slither_results = json.load(open(STAGE2 / "slither_results.json"))

def why(r):
    e = r["raw_stderr"] + r["raw_stdout"]
    if r["status"] == "timeout":                          return "timeout"
    if "NotConstant" in e:                                return "constant folding"
    if "AssertionError" in e:                             return "IR assertion"
    if "Unknown file:" in e or "not found: File" in e:    return "crytic-compile imports"
    if "Base constructor parameters" in e:                return "OZ v5 constructor"
    if "Failed to generate IR" in e:                      return "IR generation"
    return "other"

failures = [{**r, "why": why(r)} for r in slither_results if r["status"] != "ok"]
print(Counter(f["why"] for f in failures))
json.dump([{k: f[k] for k in ("path","source","label_type","status","why")} for f in failures],
          open(STAGE2 / "slither_failures.json", "w"), indent=2)

slither_ok = {r["path"] for r in slither_results if r["status"] == "ok"}
print(f"{len(slither_ok)} analyzed, {len(failures)} excluded")

## Haiku

→ Same files Slither managed. Temperature 0, one fixed prompt, no retrieval or few-shot.
→ 3 calls on a stratified 100-file sample to measure stability, 1 call elsewhere.
→ Batch API, 50% off. Caching is not used: Haiku 4.5 needs a 4,096-token prefix and these average ~2,500.

In [ ]:
from getpass import getpass
import anthropic, random

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")
client = anthropic.Anthropic()
HAIKU_MODEL = "claude-haiku-4-5-20251001"

SYSTEM_PROMPT = open("data/manifests/haiku_prompt.txt").read() \
    if os.path.exists("data/manifests/haiku_prompt.txt") else None
assert SYSTEM_PROMPT, "haiku_prompt.txt missing - the prompt is reported verbatim, not inlined"
print(f"{HAIKU_MODEL} | prompt {len(SYSTEM_PROMPT)} chars")

In [ ]:
def numbered(p):
    return "\n".join(f"{i}: {l}" for i, l in
                      enumerate(open(p, errors="ignore").read().split("\n"), 1))

targets = [r for r in manifest if r["path"] in slither_ok]
rng = random.Random(20260901)
vuln  = [r for r in targets if r["label_type"] == "vulnerable"]
clean = [r for r in targets if r["label_type"] == "clean"]

tri = []
bysrc = defaultdict(list)
for r in vuln: bysrc[r["source"]].append(r)
for s, g in sorted(bysrc.items()):
    k = min(max(round(50 * len(g) / len(vuln)), 1), len(g))
    tri += rng.sample(sorted(g, key=lambda r: r["path"]), k)
tri += rng.sample(sorted(clean, key=lambda r: r["path"]), 50)
tri_paths = {r["path"] for r in tri}

reqs = []
for i, r in enumerate(targets):
    src = numbered(r["stage1_path"])
    for c in range(3 if r["path"] in tri_paths else 1):
        reqs.append({"custom_id": f"f{i:04d}c{c}",
                     "params": {"model": HAIKU_MODEL, "max_tokens": 1000, "temperature": 0,
                                "system": SYSTEM_PROMPT,
                                "messages": [{"role": "user", "content": src}]}})
print(f"{len(targets)} files, {len(tri_paths)} triplicated, {len(reqs)} requests")

In [ ]:
batch = client.messages.batches.create(requests=reqs)
json.dump({"batch_id": batch.id, "triplicate": sorted(tri_paths)},
          open(STAGE2 / "haiku_batch_meta.json", "w"), indent=2)
print(batch.id, batch.processing_status)

→ Safe to interrupt. The batch runs server-side and the id is on disk. Do not re-run the cell above.

In [ ]:
bid = json.load(open(STAGE2 / "haiku_batch_meta.json"))["batch_id"]
while True:
    b = client.messages.batches.retrieve(bid)
    print(b.processing_status, b.request_counts)
    if b.processing_status == "ended": break
    time.sleep(30)

→ A file whose calls all fail to parse is logged `parse_error`, never coerced to all-false.

In [ ]:
def parse(text):
    t = text.strip()
    if t.startswith("```"):
        t = t.split("```")[1]
        t = t[4:] if t.startswith("json") else t
        t = t.strip()
    try: o = json.loads(t)
    except json.JSONDecodeError: return None
    if not all(c in o for c in CATEGORIES): return None
    out = {}
    for c in CATEGORIES:
        v = o[c]
        if not isinstance(v, dict) or "found" not in v: return None
        ls = v.get("lines") or []
        out[c] = {"found": bool(v["found"]),
                  "lines": [int(x) for x in ls if isinstance(x, (int, float))]}
    return out

def vote(vs):
    g = [v for v in vs if v]
    if not g: return None
    out = {}
    for c in CATEGORIES:
        f = sum(v[c]["found"] for v in g) > len(g)/2
        ln = next((v[c]["lines"] for v in g if v[c]["found"] and v[c]["lines"]), []) if f else []
        out[c] = {"found": f, "lines": ln}
    return out

def disagree(vs):
    g = [v for v in vs if v]
    return None if len(g) < 2 else sum(
        1 for c in CATEGORIES if len({v[c]["found"] for v in g}) > 1) / len(CATEGORIES)

acc = defaultdict(lambda: {"t": [], "i": 0, "o": 0})
for e in client.messages.batches.results(bid):
    if e.result.type != "succeeded": continue
    s = acc[int(e.custom_id[1:5])]
    m = e.result.message
    s["t"].append(m.content[0].text); s["i"] += m.usage.input_tokens; s["o"] += m.usage.output_tokens

haiku_results = []
for i, e in enumerate(targets):
    s = acc.get(i)
    base = {"path": e["path"], "source": e["source"], "label_type": e["label_type"],
            "n_calls": 3 if e["path"] in tri_paths else 1}
    if not s:
        haiku_results.append({**base, "status": "missing", "majority_verdict": None,
                              "raw_verdicts": [], "raw_texts": [], "disagreement_rate": None,
                              "input_tokens": 0, "output_tokens": 0}); continue
    vs = [parse(t) for t in s["t"]]
    mv = vote(vs)
    haiku_results.append({**base, "status": "ok" if mv else "parse_error",
                          "majority_verdict": mv, "raw_verdicts": vs, "raw_texts": s["t"],
                          "disagreement_rate": disagree(vs),
                          "input_tokens": s["i"], "output_tokens": s["o"]})

json.dump(haiku_results, open(STAGE2 / "haiku_results.json", "w"))
ti = sum(r["input_tokens"] for r in haiku_results); to = sum(r["output_tokens"] for r in haiku_results)
print(Counter(r["status"] for r in haiku_results))
print(f"tokens in {ti:,} out {to:,} | ${ti/1e6*.5 + to/1e6*2.5:.2f} at batch rates")
d = [r["disagreement_rate"] for r in haiku_results if r["disagreement_rate"]]
print(f"disagreement: {len(d)}/{len(tri_paths)} files, mean {sum(d)/len(tri_paths):.4f}")

## Scoring

→ Type-aware: a flag counts only if the category is right. Wrong category on a vulnerable file is a miss.
→ Two Slither mappings. Several mapped detectors only ever fire Low or Informational —
  `timestamp` (any `block.timestamp`), `low-level-calls` (a call exists), `reentrancy-events` (event order).
→ High/Medium is primary. It leaves `denial_of_service` and `time_manipulation` with zero detectors.

In [ ]:
DETECTOR_MAP = {
    "reentrancy-eth": "reentrancy", "reentrancy-no-eth": "reentrancy",
    "reentrancy-benign": "reentrancy", "reentrancy-events": "reentrancy",
    "reentrancy-balance": "reentrancy",
    "arbitrary-send-eth": "access_control", "arbitrary-send-erc20": "access_control",
    "suicidal": "access_control", "tx-origin": "access_control",
    "unprotected-upgrade": "access_control", "controlled-delegatecall": "access_control",
    "unchecked-transfer": "unchecked_low_level_calls",
    "unchecked-lowlevel": "unchecked_low_level_calls",
    "unchecked-send": "unchecked_low_level_calls",
    "low-level-calls": "unchecked_low_level_calls",
    "calls-loop": "denial_of_service", "costly-loop": "denial_of_service",
    "timestamp": "time_manipulation",
}
# weak-prng stays unmapped: bad_randomness is an excluded category.
# unused-return, controlled-array-length, uninitialized-state, locked-ether stay unmapped
# as a literal reading of the category definitions.

slither_results = json.load(open(STAGE2 / "slither_results.json"))
haiku_results   = json.load(open(STAGE2 / "haiku_results.json"))
sl = {r["path"]: r for r in slither_results}
hk = {r["path"]: r for r in haiku_results}
recs = {r["path"]: r for r in manifest}

def cats_of(r):
    if "categories" in r: return set(r["categories"])
    return {r["category"]} if "category" in r else set()

truth = {r["path"]: (cats_of(r) if r["label_type"] == "vulnerable" else set()) for r in manifest}

bad_rng   = {p for p, c in truth.items() if "bad_randomness" in c}
sl_failed = {r["path"] for r in slither_results if r["status"] != "ok"}
scored    = [p for p in truth if p not in bad_rng and p not in sl_failed and p in hk]
print(f"scored {len(scored)}  (excl {len(bad_rng)} bad_randomness, {len(sl_failed)} slither failures)")
print("vulnerable:", sum(1 for p in scored if truth[p]))

In [ ]:
def sl_flags(p, hm):
    out = defaultdict(list)
    try: d = json.loads(sl[p]["raw_stdout"])
    except Exception: return out
    for det in (d.get("results") or {}).get("detectors") or []:
        c = DETECTOR_MAP.get(det["check"])
        if not c or (hm and det["impact"] not in ("High", "Medium")): continue
        for el in det.get("elements") or []:
            out[c] += (el.get("source_mapping") or {}).get("lines") or []
    return out

def hk_flags(p):
    mv = hk[p]["majority_verdict"] or {}
    return {c: mv[c]["lines"] for c in CATEGORIES if mv.get(c, {}).get("found")}

FLAGS = {"slither_hm":  {p: sl_flags(p, True)  for p in scored},
         "slither_all": {p: sl_flags(p, False) for p in scored},
         "haiku":       {p: hk_flags(p)        for p in scored}}

def confusion(flags, paths):
    out = {}
    for c in CATEGORIES:
        tp = fp = fn = tn = 0
        for p in paths:
            a, q = c in truth[p], c in flags[p]
            tp, fp, fn, tn = (tp + (a and q), fp + (not a and q),
                              fn + (a and not q), tn + (not a and not q))
        P = tp/(tp+fp) if tp+fp else 0.0
        R = tp/(tp+fn) if tp+fn else 0.0
        out[c] = dict(tp=tp, fp=fp, fn=fn, tn=tn, n_pos=tp+fn, precision=P, recall=R,
                      f1=2*P*R/(P+R) if P+R else 0.0,
                      specificity=tn/(tn+fp) if tn+fp else 0.0)
    return out

def macro(sc):
    return {k: sum(sc[c][k] for c in CATEGORIES)/len(CATEGORIES)
            for k in ("precision", "recall", "f1", "specificity")}

scores = {"pooled": {}, "by_source": {}, "clean_fp": {}, "n_scored": len(scored)}
for n, f in FLAGS.items():
    sc = confusion(f, scored)
    scores["pooled"][n] = {"per_category": sc, "macro": macro(sc)}

for src in ["smartbugs", "swc", "dappscan"]:
    sub = [p for p in scored if recs[p]["source"] in (src, "clean")]
    scores["by_source"][src] = {
        "n_vuln": sum(1 for p in scored if recs[p]["source"] == src),
        **{n: macro(confusion(f, sub)) for n, f in FLAGS.items()}}

cl = [p for p in scored if not truth[p]]
for n, f in FLAGS.items():
    scores["clean_fp"][n] = {"n_clean": len(cl),
                             "any": sum(1 for p in cl if f[p]),
                             "per_category": {c: sum(1 for p in cl if c in f[p]) for c in CATEGORIES}}

for n in FLAGS:
    m = scores["pooled"][n]["macro"]
    print(f"{n:12} macro P{m['precision']:.2f} R{m['recall']:.2f} F{m['f1']:.2f}")

## Localization — SmartBugs only

→ Only SmartBugs ships per-line ground truth. SWC and DAppSCAN lines are stored but not scored.
→ Haiku's lines index the comment-stripped file; ground truth indexes the original. Block comments
  collapse lines, so the two must be aligned before comparing — otherwise Haiku scores ~0.04 by artifact.

In [ ]:
def strip_keep_lines(src):
    """Same stripper as Stage 1 but block comments emit their newlines, preserving line count."""
    out=[]; i=0; n=len(src); ins=False; q=""
    while i < n:
        c = src[i]
        if ins:
            out.append(c)
            if c == "\\" and i+1 < n: out.append(src[i+1]); i += 2; continue
            if c == q: ins = False
            i += 1; continue
        if c in ("'", '"'): ins=True; q=c; out.append(c); i+=1; continue
        if c == "/" and i+1 < n and src[i+1] == "/":
            while i < n and src[i] != "\n": i += 1
            continue
        if c == "/" and i+1 < n and src[i+1] == "*":
            i += 2
            while i+1 < n and not (src[i] == "*" and src[i+1] == "/"):
                if src[i] == "\n": out.append("\n")
                i += 1
            i += 2; continue
        out.append(c); i += 1
    return "".join(out)

def line_map(r):
    """stripped line -> original line"""
    pres = strip_keep_lines(open(r["path"], errors="ignore").read()).split("\n")
    cur  = open(r["stage1_path"], errors="ignore").read().split("\n")
    mp = {}; j = 0
    for oi, l in enumerate(pres, 1):
        if not l.strip(): continue
        while j < len(cur) and not cur[j].strip(): j += 1
        if j < len(cur) and cur[j].strip() == l.strip(): mp[j+1] = oi; j += 1
    return mp

upstream = json.load(open("data/manifests/smartbugs_vulnerabilities_upstream.json"))
ukey = {(e["name"], e["path"].split("/")[1]): e for e in upstream}

gt = defaultdict(dict)
for p in scored:
    r = recs[p]
    if r["source"] != "smartbugs": continue
    fn = os.path.basename(p)
    for c in cats_of(r):
        e = ukey.get((fn, c))
        if not e: continue
        for v in e["vulnerabilities"]:
            if v["category"] == c: gt[p].setdefault(c, set()).update(v["lines"])

def pred_lines(name, p):
    if name == "haiku":
        mp = line_map(recs[p])
        return {c: {mp.get(l, l) for l in ls} for c, ls in hk_flags(p).items()}
    return {c: set(ls) for c, ls in sl_flags(p, name == "slither_hm").items()}

sweep = {}
for t in [0, 1, 3, 5]:
    sweep[t] = {}
    for name in FLAGS:
        hit = tot = 0
        for p, cs in gt.items():
            pr = pred_lines(name, p)
            for c, gl in cs.items():
                tot += 1
                if any(abs(a-b) <= t for a in pr.get(c, set()) for b in gl): hit += 1
        sweep[t][name] = hit/tot if tot else 0.0
scores["localization"] = {"n_pairs": sum(len(c) for c in gt.values()),
                          "n_files": len(gt), "sweep": sweep}

print(f"{'tol':>5}" + "".join(f"{n:>14}" for n in FLAGS))
for t in sweep:
    print(f"  ±{t:<3}" + "".join(f"{sweep[t][n]:>13.2f}" for n in FLAGS))

## Cost

In [ ]:
sl_ok = [r for r in slither_results if r["status"] == "ok"]
ti = sum(r["input_tokens"] for r in haiku_results)
to = sum(r["output_tokens"] for r in haiku_results)
n_calls = sum(r["n_calls"] for r in haiku_results)

scores["cost"] = {
    "slither": {"total_s": sum(r["wallclock_s"] for r in slither_results),
                "mean_s": sum(r["wallclock_s"] for r in sl_ok)/len(sl_ok),
                "n": len(slither_results)},
    "haiku": {"input_tokens": ti, "output_tokens": to, "n_calls": n_calls,
              "usd_batch": ti/1e6*0.5 + to/1e6*2.5,
              "usd_standard": ti/1e6*1.0 + to/1e6*5.0,
              "usd_standard_1x": (ti/1e6*1.0 + to/1e6*5.0) * len(haiku_results)/n_calls},
}
json.dump(scores, open(STAGE3 / "scores.json", "w"), indent=2)
json.dump({"scored_paths": scored,
           "excluded_bad_randomness": sorted(bad_rng),
           "excluded_slither_failed": sorted(sl_failed)},
          open(STAGE3 / "scored_set.json", "w"), indent=2)
print(json.dumps(scores["cost"], indent=2))
print("\nsaved data/stage3_scored/")

# ADDITION

In [ ]:
import os, sys, json, time
from pathlib import Path
from collections import Counter, defaultdict
from getpass import getpass
import anthropic

os.chdir("/Users/mk/Desktop/mk/AuditMind")

STAGE2 = Path("data/stage2_results")
STAGE3 = Path("data/stage3_scored")

CATEGORIES = ["reentrancy", "access_control", "unchecked_low_level_calls",
              "denial_of_service", "time_manipulation"]

manifest        = json.load(open("data/manifests/stage1_manifest.json"))
slither_results = json.load(open(STAGE2 / "slither_results.json"))
slither_ok      = {r["path"] for r in slither_results if r["status"] == "ok"}
targets         = [r for r in manifest if r["path"] in slither_ok]

def numbered(p):
    return "\n".join(f"{i}: {l}" for i, l in
                     enumerate(open(p, errors="ignore").read().split("\n"), 1))

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")
client = anthropic.Anthropic()
HAIKU_MODEL = "claude-haiku-4-5-20251001"

print(f"{len(manifest)} in manifest, {len(targets)} targets")
print("existing results:", sorted(os.listdir(STAGE2)))

In [ ]:
FORCED_PROMPT = """You are a Solidity security analyzer. Pick the single vulnerability category
that best describes this contract's most serious flaw, or "none" if none of them apply.

The contract's pragma is shown near the top. Judge exploitability against that version. Some
patterns that are vulnerable in 0.4.x are safe by default in 0.8.x.

CATEGORIES

reentrancy
  An external call transfers control to another contract before the caller's state is updated,
  allowing the callee to re-enter and exploit stale state.
  Not this: .transfer() or .send() with the 2300 gas stipend, or a function already protected by
  a reentrancy guard modifier.

access_control
  A function that changes ownership, moves funds, or alters critical state is callable by an
  unauthorized party. Includes missing modifiers, wrong visibility, unprotected initializers,
  tx.origin authentication, and unprotected selfdestruct or delegatecall.
  Not this: a view or pure function without a modifier, or a function that is intentionally
  public and changes only the caller's own state.

unchecked_low_level_calls
  The return value of .call, .delegatecall, .staticcall, or .send is not checked, so a failed
  call proceeds silently as if it succeeded.
  Not this: a call whose return value is checked by require, assert, or an if that reverts.

denial_of_service
  Execution can be blocked or made unaffordable. Includes unbounded loops over arrays that grow
  with user input, and functions whose progress depends on an external call a participant can
  force to fail.
  Not this: a loop bounded by a fixed constant, or one over an array only the owner can extend.

time_manipulation
  block.timestamp or block.number is used for logic a miner can profitably influence, such as
  randomness, deadlines, payout eligibility, or lock expiry.
  Not this: block.timestamp used for an event log, or for a deadline measured in days where a
  few seconds of miner drift is irrelevant.

none
  None of the five applies. Choose this for contracts that are safe with respect to these five
  categories, even if they have other problems.

If more than one applies, pick the most serious one. Do not hedge.

OUTPUT

Return a single JSON object and nothing else. No prose, no markdown fences.

{"category": "reentrancy", "lines": [42]}

"category" is exactly one of: reentrancy, access_control, unchecked_low_level_calls,
denial_of_service, time_manipulation, none.
"lines" holds the line where it is triggered, and is empty when category is "none"."""

open("data/manifests/haiku_forced_prompt.txt", "w").write(FORCED_PROMPT)
print(len(FORCED_PROMPT), "chars")

In [ ]:
forced_reqs = [{
    "custom_id": f"f{i:04d}c0",
    "params": {"model": HAIKU_MODEL, "max_tokens": 200, "temperature": 0,
               "system": FORCED_PROMPT,
               "messages": [{"role": "user", "content": numbered(r["stage1_path"])}]},
} for i, r in enumerate(targets)]

fb = client.messages.batches.create(requests=forced_reqs)
json.dump({"batch_id": fb.id, "mode": "forced_choice"},
          open(STAGE2 / "haiku_forced_batch_meta.json", "w"), indent=2)
print(fb.id, len(forced_reqs), "requests")

In [ ]:
bid = json.load(open(STAGE2 / "haiku_forced_batch_meta.json"))["batch_id"]
while True:
    b = client.messages.batches.retrieve(bid)
    print(b.processing_status, b.request_counts)
    if b.processing_status == "ended": break
    time.sleep(30)